# 📓 Module 2 — Classifiers & Fields
**Estimated time: 30 minutes | Flags: 4, 5**

---

## The Scenario

TechCorp's attack surface has grown. The attacker is now running two campaigns:
1. **Brand impersonation phishing** — fake login pages mimicking known brands
2. **Endpoint compromise** — malicious process executions in your Windows fleet

Similarity alone can't handle phishing at scale. And endpoint events come as
**structured logs**, not free text. You need two new tools.

---

## Classifier Rules

A **trained classifier** knows the _decision boundary_ for a specific threat class.
Much more precise than generic SBERT similarity for well-defined threats:

```
classifier:
    $phish = "html content" threshold=0.6 classifier="tuned-sbert"
    #                        ↑ confidence  ↑ which model to use
```

Available classifiers:
- `tuned-sbert` — calibrated SBERT (no extra deps, works immediately)
- `distilbert` — DistilBERT embeddings
- Custom: implement `SemanticClassifier` ABC and register it

## Fields Rules (Sigma-style)

Structured event data (logs, alerts) needs **key:value matching**, not semantic:

```
fields:
    $evt  = EventID = "4625"          ← exact match
    $proc = process.name = /mimikatz/i ← regex on field value
    $ip   = source.ip = /^10\.0\./   ← regex on field value
```

Pass event data at match time:
```python
rules.matched(text, fields={"EventID": "4625", "process.name": "mimikatz.exe"})
```


In [ ]:
!pip install syara sentence-transformers -q
import syara
import base64

_FLAGS = {
    4: base64.b64decode('U1lBUkF7Y2w0c3MxZjEzcnNfcjF1bDN9').decode(),
    5: base64.b64decode('U1lBUkF7cjNhbF93MHJsZF9sNG9nX2QzdDNjdDEwbn0=').decode(),
}

def check_challenge(rules_obj, dataset, flag_id, min_recall=0.80, max_fpr=0.10,
                    use_fields=False):
    results = []
    for d in dataset:
        text   = d.get('text', '')
        fields = d.get('fields', None) if use_fields else None
        match  = rules_obj.match(text, fields=fields)[0]
        results.append(match)

    tp = sum(1 for r, d in zip(results, dataset) if r.matched and d['label'] == 'malicious')
    fp = sum(1 for r, d in zip(results, dataset) if r.matched and d['label'] == 'benign')
    fn = sum(1 for r, d in zip(results, dataset) if not r.matched and d['label'] == 'malicious')
    tn = sum(1 for r, d in zip(results, dataset) if not r.matched and d['label'] == 'benign')

    total_pos = tp + fn
    total_neg = fp + tn
    recall    = tp / total_pos if total_pos else 0
    precision = tp / (tp + fp)  if (tp + fp) else 0
    fpr       = fp / total_neg  if total_neg else 0

    print(f"\n📊 Results")
    print(f"   TP: {tp}/{total_pos}  FP: {fp}/{total_neg}  FN: {fn}")
    print(f"   Recall: {recall:.0%}  Precision: {precision:.0%}  FPR: {fpr:.0%}")

    missed = [d for r, d in zip(results, dataset)
              if not r.matched and d['label'] == 'malicious']
    if missed:
        print(f"\n💡 Missed ({len(missed)}):")
        for d in missed[:3]:
            print(f"   → {str(d.get('text','') or d.get('fields',''))[:80]}")

    if recall >= min_recall and fpr <= max_fpr:
        print(f"\n{'='*50}")
        print(f"✅  CHALLENGE {flag_id} PASSED!")
        print(f"🚩  FLAG: {_FLAGS[flag_id]}")
        print(f"{'='*50}")
        return True
    else:
        print(f"\n❌ Not quite!")
        if recall < min_recall: print("   💡 Catch more attacks")
        if fpr > max_fpr: print("   💡 Reduce false positives")
        return False

print("✅ Ready")

---
# 🎯 Challenge 4 — Brand Phishing Classifier (15 min)
## `🏁 Flag 4 — 150 points`

### Background

Brand impersonation phishing pages try to mimic well-known companies
(PayPal, Microsoft, your bank) to steal credentials.

From our research: a **classifier prefilter + LLM final check** reduced:
- Processing time: **12.5 hours → 14 minutes**
- Cost: **\$750 → \$13.50** (gemini-2.5-pro)

Today you'll build the classifier layer that makes this possible.

### Real benchmark (from the talk)

| Stage | Catch Rate | FPR | Time/page |
|-------|-----------|-----|----------|
| Classifier prefilter | 178/10,000 (1.78%) | 0.78% | 567 ms |
| LLM final check | 45/178 (0.45% of total) | 0.01% | 4.5 s |

The classifier does the **heavy lifting** — the LLM only sees the 1.78% that pass through.


In [ ]:
# ── Phishing Dataset ─────────────────────────────────────────────────────────
DATASET_4 = [
    # Phishing pages (simplified HTML text representations)
    {"id": 1,  "text": "Sign in to your PayPal account. Your account has been limited. Enter password to restore access. paypal-secure-login.fake.com", "label": "malicious"},
    {"id": 2,  "text": "Microsoft account security alert. Unusual sign-in detected. Verify your identity now or your account will be suspended. microsoft-account-verify.net", "label": "malicious"},
    {"id": 3,  "text": "Your Apple ID has been locked. Confirm your information to unlock. apple-id-verify.suspicious-domain.com", "label": "malicious"},
    {"id": 4,  "text": "Chase Bank: Your account access has been restricted. Log in immediately to prevent closure. secure.chase-bank-alert.phish", "label": "malicious"},
    {"id": 5,  "text": "Google Account suspended. Action required within 24 hours. Click here to verify your identity. google-accounts.fake-verify.net", "label": "malicious"},
    {"id": 6,  "text": "Your Amazon account has been compromised. Enter your credentials to secure your account. amazon-security-alert.com", "label": "malicious"},
    {"id": 7,  "text": "DHL Delivery Failed. Your package is on hold. Pay the customs fee of $1.99 to release your shipment. dhl-delivery-fee.phish", "label": "malicious"},
    {"id": 8,  "text": "IRS: You have a pending tax refund of $3,200. Claim your refund by verifying your Social Security Number. irs-refund-claim.net", "label": "malicious"},
    {"id": 9,  "text": "LinkedIn: Your account appears to have been accessed from an unusual location. Verify now to protect your profile.", "label": "malicious"},
    {"id": 10, "text": "Netflix: Your payment method has expired. Update your billing information to continue your subscription. netflix-billing-update.fake", "label": "malicious"},
    # Benign
    {"id": 11, "text": "Welcome to our platform. Thank you for creating your account. Explore our features today.", "label": "benign"},
    {"id": 12, "text": "Your subscription has been renewed successfully. Your next billing date is next month.", "label": "benign"},
    {"id": 13, "text": "Password reset requested. Click the link below to set a new password. If you did not request this, ignore this email.", "label": "benign"},
    {"id": 14, "text": "Security tip: Never share your password with anyone. Enable two-factor authentication for added security.", "label": "benign"},
    {"id": 15, "text": "Your order has been shipped. Track your delivery using the link below.", "label": "benign"},
    {"id": 16, "text": "Learn about phishing attacks and how to protect yourself online.", "label": "benign"},
    {"id": 17, "text": "Two-factor authentication code: 847392. This code expires in 10 minutes.", "label": "benign"},
    {"id": 18, "text": "Your account statement for this month is now available. Log in to view it.", "label": "benign"},
    {"id": 19, "text": "Reminder: Your annual subscription renews on the 15th. No action required.", "label": "benign"},
    {"id": 20, "text": "We noticed you changed your email address. If this was you, no action is needed.", "label": "benign"},
]

print(f"Phishing dataset: {sum(1 for d in DATASET_4 if d['label']=='malicious')} phishing, "
      f"{sum(1 for d in DATASET_4 if d['label']=='benign')} benign")

In [ ]:
# ── WORKED EXAMPLE — classifier basics ───────────────────────────────────────
demo_cls = syara.compile_string(r'''
    rule demo_classifier : security {
        classifier:
            # tuned-sbert: calibrated similarity classifier
            # threshold: confidence score (0.0 - 1.0)
            $cls1 = "phishing login page steal credentials" threshold=0.55 classifier="tuned-sbert"
        condition:
            $cls1
    }
''')

for item in DATASET_4[:3]:
    m = demo_cls.match(item['text'])[0]
    if m.matched and '$cls1' in m.matched_patterns and m.matched_patterns['$cls1']:
        score = m.matched_patterns['$cls1'][0].score
        print(f"🚨 [{score:.2f}] {item['text'][:65]}...")
    else:
        print(f"✅ [----] {item['text'][:65]}...")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🏁 CHALLENGE 4 — Phishing classifier                       ║
# ║                                                              ║
# ║  Goal: ≥80% recall, ≤5% FPR                                 ║
# ║                                                              ║
# ║  Hints:                                                      ║
# ║  • Try different seed phrases for classifier                 ║
# ║  • Combine with strings for known brand names               ║
# ║  • Use AND to reduce FPs: strings AND classifier            ║
# ║  • Try threshold between 0.50 and 0.70                      ║
# ╚══════════════════════════════════════════════════════════════╝

my_rules_4 = syara.compile_string(r'''
    rule phishing_detector : security phishing {
        meta:
            description = "Brand impersonation phishing detection"

        strings:
            # Urgency/suspension language common in phishing
            $urgent1 = "account has been" nocase
            $urgent2 = "action required" nocase
            $urgent3 = "TODO" nocase

        classifier:
            # Semantic classifier for phishing intent
            $phish = "TODO — write a descriptive seed phrase" threshold=0.55 classifier="tuned-sbert"

        condition:
            # Tip: try ($urgent1 or $urgent2) and $phish
            any of them
    }
''')

check_challenge(my_rules_4, DATASET_4, flag_id=4, min_recall=0.80, max_fpr=0.05)

> **💡 Key insight**: A classifier seed phrase should describe the **threat behavior**,
> not just keywords. Compare:
> - ❌ `"paypal microsoft apple"` — too specific to known brands
> - ✅ `"brand impersonation phishing login page steal credentials"` — generalizes

---
# 🎯 Challenge 5 — Endpoint Detection with Fields (15 min)
## `🏁 Flag 5 — 100 points`

### New Threat: Endpoint Compromise

Your SIEM is collecting Windows Security Event Logs. The attacker has
pivoted to an endpoint — you need to correlate multiple events to catch them.

**Attack sequence:**
1. Failed logon attempts (EventID 4625) targeting admin accounts
2. Successful logon (EventID 4624)
3. Suspicious process execution (EventID 4688) — mimikatz, psexec
4. Lateral movement via SMB (port 445)

**Fields syntax:**
```
fields:
    $f1 = EventID = "4625"                          ← exact match
    $f2 = process.name = /^(mimikatz|psexec)\.exe$/i ← regex on field
    $f3 = source.ip = /^10\./                        ← starts-with pattern
    $f4 = username = "Administrator" nocase          ← case-insensitive
```

At match time, pass the event as a dict:
```python
event = {"EventID": "4688", "process.name": "mimikatz.exe", "username": "admin"}
rules.matched("", fields=event)   # text is empty — it's all in fields
```


In [ ]:
# ── Endpoint Event Dataset ────────────────────────────────────────────────────
# Each event has both a text description and structured fields
DATASET_5 = [
    # Malicious events — attack sequence
    {"id": 1,  "text": "", "fields": {"EventID": "4625", "LogonType": "3", "TargetUserName": "Administrator", "WorkstationName": "ATTACKER-PC"}, "label": "malicious"},
    {"id": 2,  "text": "", "fields": {"EventID": "4625", "LogonType": "3", "TargetUserName": "admin", "FailureReason": "Unknown user name or bad password"}, "label": "malicious"},
    {"id": 3,  "text": "", "fields": {"EventID": "4688", "process.name": "mimikatz.exe", "CommandLine": "mimikatz.exe privilege::debug sekurlsa::logonpasswords"}, "label": "malicious"},
    {"id": 4,  "text": "", "fields": {"EventID": "4688", "process.name": "psexec.exe", "CommandLine": "psexec.exe \\\\192.168.1.5 cmd.exe"}, "label": "malicious"},
    {"id": 5,  "text": "", "fields": {"EventID": "4688", "process.name": "wce.exe", "ParentProcess": "cmd.exe"}, "label": "malicious"},
    {"id": 6,  "text": "lsadump::sam sekurlsa::logonpasswords", "fields": {"EventID": "4688", "process.name": "cmd.exe"}, "label": "malicious"},
    {"id": 7,  "text": "", "fields": {"EventID": "4625", "LogonType": "3", "TargetUserName": "root", "IpAddress": "10.0.0.55"}, "label": "malicious"},
    {"id": 8,  "text": "", "fields": {"EventID": "4688", "process.name": "gsecdump.exe", "SubjectUserName": "Administrator"}, "label": "malicious"},
    # Benign events
    {"id": 9,  "text": "", "fields": {"EventID": "4624", "LogonType": "2", "TargetUserName": "alice", "WorkstationName": "ALICE-LAPTOP"}, "label": "benign"},
    {"id": 10, "text": "", "fields": {"EventID": "4688", "process.name": "notepad.exe", "CommandLine": "notepad.exe C:\\docs\\report.txt"}, "label": "benign"},
    {"id": 11, "text": "", "fields": {"EventID": "4688", "process.name": "chrome.exe", "SubjectUserName": "bob"}, "label": "benign"},
    {"id": 12, "text": "", "fields": {"EventID": "4625", "LogonType": "2", "TargetUserName": "charlie", "FailureReason": "Wrong password"}, "label": "benign"},
    {"id": 13, "text": "", "fields": {"EventID": "4624", "LogonType": "3", "TargetUserName": "david", "IpAddress": "10.0.0.10"}, "label": "benign"},
    {"id": 14, "text": "", "fields": {"EventID": "4688", "process.name": "python.exe", "CommandLine": "python.exe script.py"}, "label": "benign"},
    {"id": 15, "text": "", "fields": {"EventID": "4688", "process.name": "powershell.exe", "CommandLine": "powershell.exe Get-Process"}, "label": "benign"},
]

print(f"Endpoint dataset: {sum(1 for d in DATASET_5 if d['label']=='malicious')} malicious, "
      f"{sum(1 for d in DATASET_5 if d['label']=='benign')} benign")
print("\nSample event (malicious):")
print(DATASET_5[2])

In [ ]:
# ── WORKED EXAMPLE — field matching ──────────────────────────────────────────
demo_fields = syara.compile_string(r'''
    rule demo_fields : endpoint {
        fields:
            $proc = process.name = /mimikatz/i
        condition:
            $proc
    }
''')

# Test with event dict
benign_event = {"EventID": "4688", "process.name": "notepad.exe"}
attack_event = {"EventID": "4688", "process.name": "mimikatz.exe"}

print(f"notepad.exe: {demo_fields.matched('', fields=benign_event)}")
print(f"mimikatz.exe: {demo_fields.matched('', fields=attack_event)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🏁 CHALLENGE 5 — Endpoint threat detection                 ║
# ║                                                              ║
# ║  Goal: ≥87% recall, ≤0% FPR (zero false alarms!)           ║
# ║                                                              ║
# ║  Hints:                                                      ║
# ║  • Use fields: for EventID, process.name                    ║
# ║  • Add strings: for cmd-line indicators (lsadump, sekurlsa) ║
# ║  • Combine: (EventID 4688 AND suspicious process) OR        ║
# ║             (EventID 4625 AND admin user AND network logon) ║
# ║  • Regex tip: /^(mimikatz|psexec|wce|gsecdump)\.exe$/i      ║
# ╚══════════════════════════════════════════════════════════════╝

my_rules_5 = syara.compile_string(r'''
    rule endpoint_threat : endpoint security {
        meta:
            description = "Endpoint compromise detection from event logs"

        strings:
            # Attacker tool command-line indicators
            $cmd1 = "lsadump" nocase
            $cmd2 = "sekurlsa" nocase
            $cmd3 = "TODO" nocase

        fields:
            # Process execution of known attacker tools
            $proc = process.name = /^(mimikatz|psexec|wce)\.(exe)$/i
            # Failed network logon targeting privileged accounts
            $evt  = EventID = "TODO"
            $user = TargetUserName = /^(admin|administrator|root)$/i

        condition:
            # Try: $proc or (any of $cmd*) or ($evt and $user)
            any of them
    }
''')

check_challenge(my_rules_5, DATASET_5, flag_id=5, min_recall=0.87, max_fpr=0.0,
                use_fields=True)

---
## 📊 Module 2 Recap

| Rule Type | Best For | Key Param |
|-----------|----------|----------|
| `classifier:` | Known threat categories (phishing, spam, malware) | `threshold=` |
| `fields:` | Structured logs / event data (SIEM, EDR) | field name + value |

**Design pattern reminder:**
```
# High-recall cheap prefilter → high-precision expensive check
condition: $strings_fast and $classifier_expensive
```

This is how we went from **\$750 → \$13.50** and **12.5 hours → 14 minutes**.

---

→ **Open `03_decision_and_llm.ipynb` for Flags 6 and 7!**
